# Regression Modeling

Use this notebook for continuous-target prediction tasks on structured data, such as sales, revenue, spend, demand, or order value. It compares baseline and stronger models and includes SHAP-based feature interpretation for tree models.


## Recommended Modeling Mindset

- Start with a simple baseline before a complex model
- Keep preprocessing inside a reproducible pipeline
- Compare models on the metric that matches the business goal
- Use interpretability tools for stakeholder-facing narratives
- Do not use RNNs for ordinary tabular data unless each observation is a true ordered sequence


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')


In [ ]:
DATA_PATH = Path('data.csv')

REGRESSION_TARGET = 'order_value'
BINARY_TARGET = 'converted'
MULTICLASS_TARGET = 'segment'
DROP_COLUMNS = ['customer_id']

df = pd.read_csv(DATA_PATH)
display(df.head())
print(df.shape)


In [ ]:
def build_preprocessor(feature_df):
    numeric_features = feature_df.select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in feature_df.columns if c not in numeric_features]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('encoder', OneHotEncoder(handle_unknown='ignore')),
                ]),
                categorical_features,
            ),
        ],
        remainder='drop',
    )
    return preprocessor, numeric_features, categorical_features


def prepare_xy(df, target_col):
    model_df = df.dropna(subset=[target_col]).copy()
    feature_cols = [c for c in model_df.columns if c not in set(DROP_COLUMNS + [target_col])]
    X = model_df[feature_cols].copy()
    y = model_df[target_col].copy()
    return X, y


## 1. Regression: Driver Analysis for Sales

Use this when the target is continuous, such as revenue, spend, order value, or weekly sales.

Suggested default comparison set:
- Linear Regression: simple and explainable baseline
- Random Forest Regressor: nonlinear interactions with little feature engineering
- XGBoost Regressor: usually strong for structured tabular data
- Optional additions: Lasso, Elastic Net, LightGBM, CatBoost


In [ ]:
X_reg, y_reg = prepare_xy(df, REGRESSION_TARGET)
preprocessor_reg, _, _ = build_preprocessor(X_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

regression_models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(n_estimators=300, random_state=42),
}

if XGBOOST_AVAILABLE:
    regression_models['xgboost'] = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
    )

regression_results = []
fitted_regression_models = {}

for name, estimator in regression_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor_reg),
        ('model', estimator),
    ])
    pipeline.fit(X_train_reg, y_train_reg)
    preds = pipeline.predict(X_test_reg)
    fitted_regression_models[name] = pipeline
    regression_results.append({
        'model': name,
        'mae': mean_absolute_error(y_test_reg, preds),
        'rmse': mean_squared_error(y_test_reg, preds, squared=False),
        'r2': r2_score(y_test_reg, preds),
    })

regression_results_df = pd.DataFrame(regression_results).sort_values('rmse')
display(regression_results_df)


In [ ]:
best_regression_model_name = regression_results_df.iloc[0]['model']
best_regression_pipeline = fitted_regression_models[best_regression_model_name]
best_regression_preds = best_regression_pipeline.predict(X_test_reg)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_reg, best_regression_preds, alpha=0.7)
ax.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], color='black')
ax.set_title(f'Actual vs Predicted: {best_regression_model_name}')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
plt.tight_layout()
plt.show()


In [ ]:
if SHAP_AVAILABLE and best_regression_model_name in {'random_forest', 'xgboost'}:
    transformed_X_train = best_regression_pipeline.named_steps['preprocessor'].fit_transform(X_train_reg)
    transformed_X_test = best_regression_pipeline.named_steps['preprocessor'].transform(X_test_reg)
    model = best_regression_pipeline.named_steps['model']
    explainer = shap.Explainer(model)
    shap_values = explainer(transformed_X_test)
    shap.plots.beeswarm(shap_values, max_display=15)
else:
    print('SHAP is available only when shap is installed and the selected model is tree-based.')


## Interpreting Results

For the selected regression model, summarize:

- Why this model was selected
- Which evaluation metric made it the winner
- Which features appear to drive the target most strongly
- What business action the team could take
- What risks remain, such as leakage, weak sample size, or unstable feature importance
